# Silver Layer — Clean & Conform Transportation Data
Dedupe, cast timestamps, derive vehicle age. NO post-trip leakage columns added.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_timestamp, year, current_date,
    row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Clean vehicles
df_v = spark.read.format('delta').table('bronze_vehicles')
w = Window.partitionBy('vehicle_id').orderBy(col('ingestion_timestamp').desc())
df_v = (
    df_v.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('capacity_tonnes', col('capacity_tonnes').cast('double'))
    .withColumn('year_registered', col('year_registered').cast('int'))
    .withColumn('vehicle_age', year(current_date()) - col('year_registered'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_v.write.mode('overwrite').format('delta').saveAsTable('silver_vehicles')
print(f'silver_vehicles: {df_v.count()} rows')

In [ ]:
# Clean routes
df_r = spark.read.format('delta').table('bronze_routes')
w2 = Window.partitionBy('route_id').orderBy(col('ingestion_timestamp').desc())
df_r = (
    df_r.withColumn('_rn', row_number().over(w2)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('distance_km', col('distance_km').cast('double'))
    .withColumn('sla_hours', col('sla_hours').cast('double'))
    .withColumn('toll_cost_gbp', col('toll_cost_gbp').cast('double'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_r.write.mode('overwrite').format('delta').saveAsTable('silver_routes')
print(f'silver_routes: {df_r.count()} rows')

In [ ]:
# Clean deliveries (keep label is_late; downstream FE drops post-trip leakage)
df_d = spark.read.format('delta').table('bronze_deliveries')
w3 = Window.partitionBy('delivery_id').orderBy(col('ingestion_timestamp').desc())
df_d = (
    df_d.withColumn('_rn', row_number().over(w3)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('planned_departure', to_timestamp('planned_departure'))
    .withColumn('planned_duration_hrs', col('planned_duration_hrs').cast('double'))
    .withColumn('distance_km', col('distance_km').cast('double'))
    .withColumn('load_tonnes', col('load_tonnes').cast('double'))
    .withColumn('is_late', col('is_late').cast('int'))
    .filter(col('planned_departure').isNotNull())
    .withColumn('silver_timestamp', current_timestamp())
)
df_d.write.mode('overwrite').format('delta').saveAsTable('silver_deliveries')
print(f'silver_deliveries: {df_d.count()} rows')

In [ ]:
# Clean fuel logs
df_f = spark.read.format('delta').table('bronze_fuel_logs')
df_f = (
    df_f
    .withColumn('litres_filled', col('litres_filled').cast('double'))
    .withColumn('total_cost_gbp', col('total_cost_gbp').cast('double'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_f.write.mode('overwrite').format('delta').saveAsTable('silver_fuel_logs')
print(f'silver_fuel_logs: {df_f.count()} rows')